In [54]:
2+2


4

In [55]:
from langchain_groq import ChatGroq 
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings # Isko change kiya
from langchain_community.vectorstores import FAISS

In [56]:
def file_loader(path):
  loader = DirectoryLoader(
    path, glob="*.pdf", loader_cls=PyPDFLoader
  )
  documents = loader.load()
  return documents

In [57]:
extracted_docs = file_loader(r"Data/")

In [58]:
def chunking_data(data):
  split_data = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 50)
  chunk_data = split_data.split_documents(data)
  return chunk_data

In [59]:
chunk_data = chunking_data(extracted_docs)
len(chunk_data)

2124

In [60]:
chunk_data[40].page_content

'backs (including the Tensor Board callback). • Chapter 11 (many changes)\n— Introduced self-normalizing nets, the SELU activation function and Alpha\nDropout. — Introduced self-supervised learning. — Added Nadam optimiza-\ntion. — Added Monte-Carlo Dropout. — Added a note about the risks of\nadaptive optimization methods. — Updated the practical guidelines. • Chap-\nter 12 – completely rewritten chapter, including: — A tour of TensorFlow 2 —'

In [61]:
def get_embedding():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [62]:
embedding = get_embedding()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3618.17it/s]


In [63]:
docs = FAISS.from_documents(documents=chunk_data, embedding=embedding )
docs

In [64]:
retriver = docs.as_retriever(search_type="similarity", search_kwargs={"k": 3})
output = retriver.invoke("What is Supervised Machine Learning")
output

[Document(id='0f93b77b-76bd-4ae0-8d33-c982512f01f1', metadata={'producer': 'xdvipdfmx (20211117)', 'creator': 'LaTeX via pandoc', 'creationdate': '2025-01-23T13:03:17+00:00', 'source': 'Data\\Hands-On-Machine-Learning.pdf', 'total_pages': 290, 'page': 11, 'page_label': '12'}, page_content='Figure 1-5. A labeled training set for supervised learning (e.g., spam classifica-\ntion) 8 | Chapter 1: The Machine Learning Landscape\nA typical supervised learning task is classification. The spam filter is a good\nexample of this: it is trained with many example emails along with their class\n(spam or ham), and it must learn how to classify new emails. Another typical\ntask is to predict a target numeric value, such as the price of a car, given a set of'),
 Document(id='ead6e988-5964-481b-b40e-ed1b9c03e49a', metadata={'producer': 'xdvipdfmx (20211117)', 'creator': 'LaTeX via pandoc', 'creationdate': '2025-01-23T13:03:17+00:00', 'source': 'Data\\Hands-On-Machine-Learning.pdf', 'total_pages': 290, 

In [65]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    temperature=0.6,
    model_name="llama-3.1-8b-instant",  # ya "mixtral-8x7b-32768"
    
)

In [66]:
%pip install langchain langchain-community langchain-groq


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [68]:
system_prompt = (
"You are an expert Data Scientist assistat of qestion-answering tasks."
"Use the following pieces of retrieved context to answer "
"the question. If you don't find any related context then say that you "
"don't know. Do not give any halusinating answer of this. Use the three sentece maximum and keep the "
"answer concise."
"\n\n"
"{context}"
 )

chat_prompt = ChatPromptTemplate.from_messages([
  ("system", system_prompt),
  ("user", "{input}" )]
)

In [69]:
stuff_chain =create_stuff_documents_chain(llm, chat_prompt)
retriver_chain = create_retrieval_chain(retriver, stuff_chain)
question = "What is Machine Learnig?"
response_dict = retriver_chain.invoke({"input" : question})
# response = response_dict["answer"] if isinstance(response_dict, dict) else str(response_dict)
response = response_dict['answer']
response

'Machine Learning is a field of study that focuses on developing algorithms and statistical models that enable computers to learn from data, allowing them to make predictions, classify objects, and make decisions without being explicitly programmed. This means that a machine can improve its performance on a task over time based on the data it receives.'

In [70]:
question = "What is Tensorflow?"
response_dict = retriver_chain.invoke({"input" : question})
# response = response_dict["answer"] if isinstance(response_dict, dict) else str(response_dict)
response = response_dict['answer']
response

'TensorFlow is an open-source machine learning library that offers various functionalities such as language processing, recommender systems, time series forecasting, and more. Its core is similar to NumPy but with GPU support and it includes just-in-time (JIT) compilation for optimization.'